[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-09-custom-pyfunc.ipynb#scrollTo=aa1bb2cc)

---
# Day 9 · Custom Python Function Models and Preprocessing Pipelines
**certified-journeys / mlflow-certified** · Practice · Custom Model Packaging

> **Goal for today:** Build a custom `mlflow.pyfunc.PythonModel` that bundles a `StandardScaler` preprocessing step with a trained model, log it with artifacts, load it back, and run end-to-end inference on raw data.


In [ ]:
%pip install -q mlflow scikit-learn pandas numpy joblib


## Step 1 · Why PyFunc? The Universal MLflow Model Format

MLflow ships with built-in **model flavors** for sklearn, PyTorch, TensorFlow, XGBoost, etc. But what if your model:
- Uses a custom framework not supported out of the box?
- Needs preprocessing baked in (so callers pass raw features, not scaled ones)?
- Runs a business rule ensemble combining multiple sub-models?

**PyFunc** (`mlflow.pyfunc`) is the answer. It defines a single interface:

| Interface | Description |
|---|---|
| `predict(context, model_input)` | Called at inference time; `model_input` is a DataFrame or numpy array |
| `context.artifacts` | Dict mapping name → local path of any file you bundled at log time |
| `PythonModel` base class | Extend this and override `predict` (and optionally `load_context`) |

PyFunc models get serving, signature validation, and Model Registry support for free — the same as any built-in flavor.


In [ ]:
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import numpy as np
import pandas as pd
import joblib
import os
import tempfile
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Local tracking URI
mlflow.set_tracking_uri("sqlite:///mlflow_pyfunc_demo.db")
mlflow.set_experiment("day-09-custom-pyfunc")

# Load data — breast cancer is a binary classification problem
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Features: {X.shape[1]}, Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"Feature names (first 5): {list(X.columns[:5])}")


**What just happened?**
- We're using the **breast cancer dataset** (30 continuous features) to demonstrate why preprocessing matters — the features have very different scales (area vs smoothness vs texture).
- A `StandardScaler` that wasn't applied to training data would be useless at serving time — it must be bundled with the model.
- This is the core problem PyFunc solves: keeping preprocessing and model weights together as a single deployable unit.


## Step 2 · Train the Scaler and Model Separately

Before building the PyFunc wrapper, we train both components independently so we can examine them. At log time, we'll serialize both to disk and pass them as artifacts.

| Component | Serialization | Why |
|---|---|---|
| `StandardScaler` | `joblib.dump` | Captures `mean_` and `scale_` fitted values |
| `RandomForestClassifier` | `joblib.dump` | Captures all decision trees |

We could use `pickle` instead of `joblib`, but `joblib` is more efficient for numpy-heavy objects and is the scikit-learn standard.


In [ ]:
# Fit scaler on training data only (never fit on test data — data leakage!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # transform only, no fit

# Train classifier on scaled features
clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf.fit(X_train_scaled, y_train)

# Evaluate standalone performance
preds = clf.predict(X_test_scaled)
proba = clf.predict_proba(X_test_scaled)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, proba):.4f}")

# Serialize both to a temp directory (we'll pass these as artifacts)
artifact_dir = tempfile.mkdtemp()
scaler_path = os.path.join(artifact_dir, "scaler.pkl")
model_path  = os.path.join(artifact_dir, "model.pkl")

joblib.dump(scaler, scaler_path)
joblib.dump(clf,    model_path)

print(f"\nSerializer files:")
print(f"  scaler: {scaler_path} ({os.path.getsize(scaler_path):,} bytes)")
print(f"  model:  {model_path}  ({os.path.getsize(model_path):,} bytes)")


**What just happened?**
- **`scaler.fit_transform(X_train)`** fits the scaler (computes `mean_` and `scale_`) AND transforms in one step — only call this on training data.
- **`scaler.transform(X_test)`** applies the *already-fitted* scaler to test data — this is how you prevent leakage.
- `tempfile.mkdtemp()` creates an isolated directory for our artifact files; MLflow will copy these into the run's artifact store.


## Step 3 · Create a `PythonModel` Class

To create a custom PyFunc model:
1. Extend `mlflow.pyfunc.PythonModel`
2. Optionally override `load_context(context)` — called once at load time to initialize heavy state
3. Override `predict(context, model_input, params=None)` — called at every inference request

The `context` object has:
- **`context.artifacts`** — dict mapping artifact names to their local paths (populated from the `artifacts` dict you pass at log time)
- **`context.model_config`** — free-form dict of model configuration


In [ ]:
class ScaledClassifier(mlflow.pyfunc.PythonModel):
    """Custom PyFunc model that bundles StandardScaler + RandomForestClassifier.
    
    The scaler and model are loaded from artifacts at inference time,
    so callers pass raw (unscaled) features — preprocessing is automatic.
    """

    def load_context(self, context):
        """Called once when the model is loaded. Use for expensive initialization."""
        # context.artifacts maps the name we chose to the file's local path
        self.scaler = joblib.load(context.artifacts["scaler"])
        self.clf    = joblib.load(context.artifacts["classifier"])

    def predict(self, context, model_input, params=None):
        """Main inference method.
        
        Args:
            context: PythonModelContext with artifacts and model_config
            model_input: pd.DataFrame or np.ndarray of raw (unscaled) features
            params: optional dict of runtime params (MLflow 2.x feature)
        
        Returns:
            pd.DataFrame with columns 'prediction' and 'probability'
        """
        # Accept both DataFrame and numpy array inputs
        if isinstance(model_input, pd.DataFrame):
            X = model_input.values
        else:
            X = np.array(model_input)

        # Apply the bundled scaler — callers never need to know about scaling
        X_scaled = self.scaler.transform(X)

        # Optional runtime param: return_proba (default False)
        return_proba = (params or {}).get("return_proba", False)

        preds = self.clf.predict(X_scaled)
        proba = self.clf.predict_proba(X_scaled)[:, 1]  # probability of class 1

        result = pd.DataFrame({"prediction": preds, "probability": proba})
        return result if return_proba else result[["prediction"]]

print("ScaledClassifier class defined.")
print("Methods:", [m for m in dir(ScaledClassifier) if not m.startswith("_")])


**What just happened?**
- **`load_context`** runs once at model load — `joblib.load` is the expensive I/O operation, and doing it here avoids re-loading on every `predict` call.
- **`context.artifacts["scaler"]`** resolves the `"scaler"` key to the file path where MLflow copied the artifact — the key must match exactly what we pass in the `artifacts` dict at log time.
- **`params`** (MLflow 2.x) allows callers to pass runtime flags like `return_proba=True` without changing the model signature.


## Step 4 · Log the Custom PyFunc Model with `mlflow.pyfunc.log_model`

The `log_model` call for PyFunc has a few important parameters beyond the standard ones:

| Parameter | Required | Description |
|---|---|---|
| `python_model` | Yes | Instance or class of your `PythonModel` |
| `artifact_path` | Yes | Subdirectory name in the run's artifacts |
| `artifacts` | No | `{name: local_path}` — files to bundle with the model |
| `input_example` | Recommended | Auto-infers model signature |
| `conda_env` | No | Environment spec; defaults to current env if omitted |
| `code_paths` | No | Additional Python files to bundle (e.g. helper modules) |


In [ ]:
from mlflow.models.signature import infer_signature

with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 6,
        "preprocessing": "StandardScaler",
    })

    # Log metrics (computed earlier)
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, preds),
        "roc_auc":  roc_auc_score(y_test, proba),
    })

    # Infer signature from raw (unscaled) inputs — the model's public interface
    signature = infer_signature(
        X_test,                            # raw feature DataFrame
        pd.DataFrame({"prediction": preds}) # expected output shape
    )

    # Log the custom pyfunc model
    # artifacts dict: map the name used in load_context to the local file path
    mlflow.pyfunc.log_model(
        artifact_path="scaled-classifier",
        python_model=ScaledClassifier(),
        artifacts={
            "scaler":     scaler_path,   # → context.artifacts["scaler"]
            "classifier": model_path,    # → context.artifacts["classifier"]
        },
        input_example=X_test.head(3),
        signature=signature,
    )

    logged_run_id = run.info.run_id

print(f"Model logged. run_id={logged_run_id}")
print(f"Artifact URI: {mlflow.get_artifact_uri()}")


**What just happened?**
- **`artifacts` dict** maps string keys (used inside `load_context`) to local file paths — MLflow copies these files into the run artifact store.
- **`infer_signature`** inspects the raw input DataFrame and the output to build a Schema — MLflow uses this at serving time to validate request payloads.
- The `input_example` is stored alongside the model and used by MLflow serving to generate curl examples automatically.


## Step 5 · Load the Custom Model and Run Inference

Loading a PyFunc model gives you a **generic Python function wrapper** — you call `.predict()` with a DataFrame or numpy array and get back a DataFrame. You never need to know whether the underlying model is a `RandomForest`, a neural net, or a rule engine.

This uniformity is what makes PyFunc the foundation for MLflow's model serving:
- `mlflow models serve` calls `.predict(context, input_df)`
- The MLflow REST scoring server calls `.predict(context, input_df)`
- Spark UDFs created from PyFunc call `.predict(context, input_df)`


In [ ]:
# Load the model using the runs:/ URI
model_uri = f"runs:/{logged_run_id}/scaled-classifier"
loaded_model = mlflow.pyfunc.load_model(model_uri)

print("Model loaded. Type:", type(loaded_model))
print("Flavor:", loaded_model.metadata.flavors.keys() if loaded_model.metadata else "N/A")

# Run inference on raw (unscaled) test data
# The model handles scaling internally — caller passes raw features
predictions = loaded_model.predict(X_test.head(10))

print("\nFirst 10 predictions (raw input → scaled internally → classified):")
print(predictions)

# Verify accuracy matches what we computed before logging
all_preds = loaded_model.predict(X_test)
loaded_accuracy = accuracy_score(y_test, all_preds["prediction"])
print(f"\nAccuracy via loaded PyFunc model: {loaded_accuracy:.4f}")


**What just happened?**
- **`mlflow.pyfunc.load_model(uri)`** reconstructed the `ScaledClassifier` by calling `load_context` with the artifacts path — `scaler.pkl` and `model.pkl` were loaded into memory.
- We passed **raw, unscaled data** to `.predict()` and got correct results — the preprocessing is invisible to the caller.
- The returned object is a `PyFuncModel` wrapper, not our class directly — this abstraction is what enables serving.


## Step 6 · Use Runtime `params` for Flexible Output

MLflow 2.x introduced **inference-time parameters** — extra arguments passed to `predict()` that control behavior without changing the model artifact. This is useful for:
- Returning probabilities vs hard labels
- Selecting a threshold for binary classification
- Choosing between multiple sub-models bundled in one PyFunc

Params are declared in the model signature and can be passed at load or predict time.


In [ ]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, ParamSchema, ParamSpec
from mlflow.types import DataType

# Build a signature that declares the 'return_proba' runtime param
input_schema = Schema([ColSpec(DataType.double, name=col) for col in X.columns])
output_schema = Schema([ColSpec(DataType.long, "prediction"), ColSpec(DataType.double, "probability")])
param_schema = ParamSchema([ParamSpec("return_proba", DataType.boolean, default=False)])

sig_with_params = ModelSignature(
    inputs=input_schema,
    outputs=output_schema,
    params=param_schema,
)

# Log a second version with the params-aware signature
with mlflow.start_run() as run2:
    mlflow.log_params({"version": "v2-with-params", "n_estimators": 100})
    mlflow.pyfunc.log_model(
        artifact_path="scaled-classifier-v2",
        python_model=ScaledClassifier(),
        artifacts={"scaler": scaler_path, "classifier": model_path},
        signature=sig_with_params,
    )
    run2_id = run2.info.run_id

# Load and call with return_proba=True
model_v2 = mlflow.pyfunc.load_model(f"runs:/{run2_id}/scaled-classifier-v2")

# Pass runtime param — returns both prediction and probability
result_with_proba = model_v2.predict(X_test.head(5), params={"return_proba": True})
print("Output with return_proba=True:")
print(result_with_proba)

# Without param — returns only prediction column (default behavior)
result_default = model_v2.predict(X_test.head(5))
print("\nOutput with default params (return_proba=False):")
print(result_default)


**What just happened?**
- **`ParamSchema`** declares typed runtime parameters with defaults — MLflow validates incoming params against this schema.
- **`params={"return_proba": True}`** at predict time flows through to our `predict` method's `params` argument — no model reload needed.
- This pattern lets a single deployed model serve multiple use cases (batch scoring wants labels; A/B testing code wants probabilities).


## Step 7 · Log External Artifacts and Bundle Extra Files

The `artifacts` dict can contain any file — not just model weights. Common patterns:

| Artifact key | Contents | Use in `predict` |
|---|---|---|
| `scaler` | Fitted scaler `.pkl` | `context.artifacts["scaler"]` |
| `label_encoder` | Fitted label encoder | Decode numeric outputs to class names |
| `feature_config` | JSON with feature engineering rules | Apply before calling sub-model |
| `threshold_file` | JSON with optimal thresholds per class | Override default 0.5 cutoff |
| `vocabulary` | Text vectorizer vocabulary | NLP preprocessing |


In [ ]:
import json

# Create a threshold config file — an example of bundling non-model artifacts
threshold_config = {
    "binary_threshold": 0.45,  # tuned for recall vs precision trade-off
    "class_names": ["malignant", "benign"],
    "feature_count": X.shape[1],
    "trained_on": "breast_cancer_sklearn",
}

config_path = os.path.join(artifact_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(threshold_config, f, indent=2)

print("Config file contents:")
print(open(config_path).read())


class ThresholdClassifier(mlflow.pyfunc.PythonModel):
    """Extended PyFunc model that uses a bundled threshold config file."""

    def load_context(self, context):
        self.scaler = joblib.load(context.artifacts["scaler"])
        self.clf    = joblib.load(context.artifacts["classifier"])
        # Load the JSON config from artifacts
        with open(context.artifacts["config"]) as f:
            self.config = json.load(f)

    def predict(self, context, model_input, params=None):
        if isinstance(model_input, pd.DataFrame):
            X = model_input.values
        else:
            X = np.array(model_input)

        X_scaled = self.scaler.transform(X)
        proba = self.clf.predict_proba(X_scaled)[:, 1]

        # Apply the bundled threshold (not the default 0.5)
        threshold = self.config["binary_threshold"]
        preds = (proba >= threshold).astype(int)

        # Map numeric predictions to class names
        class_names = self.config["class_names"]
        labels = [class_names[p] for p in preds]

        return pd.DataFrame({
            "prediction": preds,
            "label": labels,
            "probability": np.round(proba, 4),
        })


# Log this richer model
with mlflow.start_run() as run3:
    mlflow.pyfunc.log_model(
        artifact_path="threshold-classifier",
        python_model=ThresholdClassifier(),
        artifacts={
            "scaler":     scaler_path,
            "classifier": model_path,
            "config":     config_path,   # bundled JSON config
        },
        input_example=X_test.head(2),
    )
    run3_id = run3.info.run_id

# Load and test
model_v3 = mlflow.pyfunc.load_model(f"runs:/{run3_id}/threshold-classifier")
result_v3 = model_v3.predict(X_test.head(6))
print("\nThresholdClassifier output:")
print(result_v3)


**What just happened?**
- We bundled a **JSON config file** alongside the model artifacts — no separate config service or environment variable needed.
- `context.artifacts["config"]` gives the local path to the JSON file after MLflow copies it to the serving environment.
- This pattern is ideal for threshold tuning: update the config JSON and re-log the model without retraining — a much cheaper operation.


## Step 8 · Register the PyFunc Model in the Registry

A PyFunc model is registered exactly like any other MLflow model — the Registry doesn't care about the flavor. This means all Day 7 Registry workflows (stages, aliases, `load_model` by alias) work identically.


In [ ]:
from mlflow.tracking import MlflowClient

PYFUNC_MODEL_NAME = "breast-cancer-classifier"

# Register the threshold classifier model
model_uri = f"runs:/{run3_id}/threshold-classifier"
registered = mlflow.register_model(model_uri=model_uri, name=PYFUNC_MODEL_NAME)

print(f"Registered: {registered.name} v{registered.version}")

# Set a 'champion' alias
client = MlflowClient()
client.set_registered_model_alias(
    name=PYFUNC_MODEL_NAME,
    alias="champion",
    version=registered.version,
)

# Load from registry by alias — same URI format as sklearn, pytorch, etc.
champion = mlflow.pyfunc.load_model(f"models:/{PYFUNC_MODEL_NAME}@champion")
champion_preds = champion.predict(X_test.head(3))
print("\nLoaded from registry by @champion alias:")
print(champion_preds)


**What just happened?**
- **`mlflow.register_model`** works identically for PyFunc — the Registry is flavor-agnostic.
- **`models:/<name>@champion`** resolved to the `ThresholdClassifier` PyFunc model — the caller uses `mlflow.pyfunc.load_model`, not `mlflow.sklearn.load_model`.
- This is the final pattern: **train → log (with artifacts) → register → alias → load by alias** — a complete ML lifecycle in one notebook.


In [ ]:
# Challenge: Build a multi-model ensemble PyFunc
#
# Task: Create a class 'EnsembleModel' that extends PythonModel and:
#   1. Loads TWO pre-trained sklearn models from artifacts (e.g. rf_model and gb_model)
#   2. Loads a scaler artifact
#   3. In predict(), runs both models on the scaled input and averages their predict_proba
#   4. Returns a DataFrame with columns: 'ensemble_prediction', 'avg_probability'
#
# Hints:
#   - Train a GradientBoostingClassifier in addition to the existing RandomForest
#   - joblib.dump both models to separate files
#   - Pass both as artifacts: {"rf": rf_path, "gb": gb_path, "scaler": scaler_path}
#   - Average probabilities: (rf_proba + gb_proba) / 2
#   - Use (avg_proba >= 0.5).astype(int) for final predictions

class EnsembleModel(mlflow.pyfunc.PythonModel):
    # Your solution here
    def load_context(self, context):
        pass

    def predict(self, context, model_input, params=None):
        pass

# Test scaffold (uncomment when implemented):
# from sklearn.ensemble import GradientBoostingClassifier
# gb = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)
# gb_path = os.path.join(artifact_dir, "gb_model.pkl")
# joblib.dump(gb, gb_path)
#
# with mlflow.start_run():
#     mlflow.pyfunc.log_model(
#         artifact_path="ensemble",
#         python_model=EnsembleModel(),
#         artifacts={"rf": model_path, "gb": gb_path, "scaler": scaler_path},
#     )


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow.pyfunc.PythonModel` | Base class — override `load_context` and `predict` |
| `load_context(context)` | Called once at load; use for expensive initialization (joblib.load) |
| `context.artifacts["key"]` | Resolves artifact name to local file path at inference time |
| `artifacts={}` in `log_model` | Maps string keys to local file paths; MLflow copies files into the run |
| `mlflow.pyfunc.load_model(uri)` | Loads any PyFunc model — flavor-agnostic |
| `params={}` in `predict` | MLflow 2.x: runtime behavior flags without reloading the model |
| Registry + PyFunc | `mlflow.register_model` and alias/stage APIs work identically for PyFunc |

> **Tip:** PyFunc is the universal MLflow model format — if your model isn't a standard library, wrap it in a PythonModel and you get serving, signature validation, and registry support for free.

---
## What's next
**Day 10** → MLflow Model Serving — deploy a registered model as a REST endpoint with `mlflow models serve`, validate requests against the signature, and test with curl.

Mark Day 9 complete in your [tracker](../index.html).
